# S0E2 lab · One reach, two families

**Video:** S0E2 · The four families of action models on [Foundation Models for Robotics](https://www.youtube.com/@FoundationModelsForRobotics)
**Book:** [Action Models for Robot Learning, §1.4](https://action-models-book.vercel.app/chapters/1/4/)
**You need:** Lab Kit L1 (or just Python). No GPU. About 20 minutes.

The task: put the FR3's fingertips on a point above the table, hand pointing down. You'll solve it with a **classical** action model (inverse kinematics, derived from the arm's geometry) and an **imitation** one (a policy that only ever sees 50 demonstrations and copies them). Then you'll move the target somewhere the demonstrations never went.

In [ ]:
# Setup: works in Google Colab and locally (after `pip install -e .` in the repo root)
import os, sys
if "google.colab" in sys.modules:
    os.environ["MUJOCO_GL"] = "egl"
    if not os.path.exists("fmr-robot-learning-labs"):
        !git clone -q https://github.com/kandapagari/fmr-robot-learning-labs.git
    %cd fmr-robot-learning-labs
    !pip install -q -e .
    sys.path.insert(0, "labs/s0e2-two-families")
else:
    sys.path.insert(0, os.path.abspath("."))
import numpy as np, matplotlib.pyplot as plt, mediapy as media
import fmr6, two_families as tf

## 1 · Fifty demonstrations, all on the left

A demonstration here is a pair: a target, and the joint angles that reach it. Real demonstrations come from a person driving the robot; we cheat and record them with IK, which is fine because the copier never gets to call IK itself. All fifty targets are on the left half of the table (y < 0).

In [ ]:
robot = fmr6.Robot("table", arm="fr3")
demo_targets = tf.sample_targets(50, tf.LEFT_Y, seed=0)
demo_angles = tf.demonstrate(robot, demo_targets)
copier = tf.CopyDemos(k=8).fit(demo_targets, demo_angles)
print(demo_targets.shape, demo_angles.shape)

## 2 · Inside the demonstrations

New targets, still on the left. Read `CopyDemos.__call__` in `two_families.py`: it finds the 8 nearest demos and fits a small linear map from target to joint angles. It knows nothing about links or joints.

In [ ]:
inside = tf.compare(robot, copier, tf.sample_targets(30, tf.LEFT_Y, seed=1))
print(f"IK:     worst {inside[:, 0].max() * 1000:.2f} mm")
print(f"copier: median {np.median(inside[:, 1]) * 1000:.1f} mm, worst {inside[:, 1].max() * 1000:.1f} mm")

Both land within a few millimetres. If you only ever tested inside the demonstrations, you couldn't tell them apart.

## 3 · Outside the demonstrations

In [ ]:
outside = tf.compare(robot, copier, tf.sample_targets(30, tf.RIGHT_Y, seed=2))
print(f"IK:     worst {outside[:, 0].max() * 1000:.2f} mm")
print(f"copier: median {np.median(outside[:, 1]) * 1000:.0f} mm, worst {outside[:, 1].max() * 1000:.0f} mm")
plt.figure(figsize=(7, 3.5))
plt.scatter(inside[:, 2] * 100, inside[:, 1] * 1000, label="inside (left half)")
plt.scatter(outside[:, 2] * 100, outside[:, 1] * 1000, label="outside (right half)")
plt.xlabel("distance to the nearest demonstration (cm)"); plt.ylabel("copier's miss (mm)"); plt.legend(); plt.tight_layout(); plt.show()

IK doesn't care where the target is, as long as the arm can reach it. The copier's error grows with distance from the nearest demonstration: it's extrapolating a pattern it only saw on the left.

## 4 · See it

In [ ]:
target = np.array([0.60, 0.22, 0.45])                     # far right
cam = {"lookat": [0.58, 0.05, 0.45], "distance": 1.3, "azimuth": 160, "elevation": -35}
pics = []
for name, q in [("IK (classical)", robot.ik(target, q0=robot.ready)), ("copier (imitation)", copier(target))]:
    robot.set_q(q)
    tip = robot.tcp()[0]
    pics.append(robot.render(420, 300, camera=cam, trail=np.array([tip, target]), guide=demo_targets[:, :3]))
    print(f"{name:20s} misses by {np.linalg.norm(tip - target) * 1000:6.1f} mm")
media.show_images(pics, titles=["IK (classical)", "copier (imitation)"])

The grey line runs through the demo targets; the red line joins the fingertips to the target. A short red line is a good reach.

## 5 · Try it

1. Add 10 demonstrations on the right (`tf.sample_targets(10, tf.RIGHT_Y, seed=3)`), refit, and rerun section 3. How much does the worst miss drop?
2. Cut the demonstrations from 50 to 5. What happens inside the left half?
3. Change `k` from 8 to 2 and to 30. Which is better inside, which outside, and why?
4. **The S0E2 point:** IK needed a model of the arm and no data. The copier needed data and no model. Which would you trust on a robot whose link lengths you don't know exactly?

In [ ]:
# your experiments here